# 🧠 Implementing the Functional API in Keras

> *Building neural networks the flexible way — defining graphs of layers, not just linear stacks.*

---

## 📋 Overview

Keras offers two main ways to build models:

| API | Style | Flexibility |
|-----|-------|-------------|
| **Sequential** | Stack layers one after another | Limited — one input, one output, linear flow |
| **Functional** | Define tensors flowing through layers | Full — any graph topology |

In this notebook I implement the **Functional API** from scratch. I build an input layer, wire up hidden layers, define an output layer, compile, train, and evaluate — then extend the model with Dropout and Batch Normalization.

**Why this matters for my background:**  
The Functional API maps naturally to signal-processing pipelines. A Dense layer is essentially a linear transform followed by a non-linearity — the same way a filter bank processes a frequency vector into a feature vector. The graph structure lets me wire up parallel branches, skip connections, and multi-input architectures — exactly the kind of flexibility needed for real RF or network data problems.

## 🧩 Theory

### The Functional API Mental Model

In the Functional API, each layer is a **function** that takes a tensor and returns a tensor. I build a model by composing these functions:

$$x_0 = \text{Input}(\text{shape})$$
$$x_1 = \text{Dense}(n_1, \phi_1)(x_0)$$
$$x_2 = \text{Dense}(n_2, \phi_2)(x_1)$$
$$\hat{y} = \text{Dense}(n_{\text{out}}, \phi_{\text{out}})(x_2)$$

Then I bind input to output:
$$\text{model} = \text{Model}(\text{inputs}=x_0,\ \text{outputs}=\hat{y})$$

### Activation Functions

| Activation | Formula | Typical Use |
|------------|---------|-------------|
| **ReLU** | $\phi(z) = \max(0, z)$ | Hidden layers (default choice) |
| **Tanh** | $\phi(z) = \tanh(z)$ | Hidden layers (zero-centered) |
| **Sigmoid** | $\phi(z) = \frac{1}{1+e^{-z}}$ | Binary classification output |

### Binary Cross-Entropy Loss

For binary classification (output ∈ {0, 1}), the loss is:

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

This penalises confident wrong predictions heavily — exactly what we want for a clean decision boundary.

## Part 1 — 🏗️ Building a Model with the Functional API

### Step 1 — Import Libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='tensorflow')

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.17.0


**What I imported:**

- `Model` — the class that wraps a computation graph into a trainable object
- `Input` — creates a symbolic entry point (like defining the antenna port on an RF system — it specifies the signal dimensions before any processing)
- `Dense` — a fully connected layer: each neuron receives every feature from the previous layer

### Step 2 — Define the Input Layer

I'm working with feature vectors of length 20 (e.g., 20 measured signal features per sample).

In [2]:
input_layer = Input(shape=(20,))
print(input_layer)

<KerasTensor shape=(None, 20), dtype=float32, sparse=False, ragged=False, name=keras_tensor>


`Input(shape=(20,))` creates a **symbolic tensor** — it doesn't hold data yet, it just describes the shape the model will expect. Think of it as the input port specification on a filter: 20-channel input, batch size unspecified (inferred at runtime).

### Step 3 — Add Hidden Layers

Two hidden layers, each with 64 units and ReLU activation. Each layer transforms the representation, learning increasingly abstract features.

$$z_1 = W_1 \cdot x_0 + b_1, \quad h_1 = \text{ReLU}(z_1)$$
$$z_2 = W_2 \cdot h_1 + b_2, \quad h_2 = \text{ReLU}(z_2)$$

In [3]:
hidden_layer1 = Dense(64, activation='relu')(input_layer)
hidden_layer2 = Dense(64, activation='relu')(hidden_layer1)

The Functional API syntax `Dense(64, activation='relu')(input_layer)` means: create a Dense layer with 64 units, then *call* it on `input_layer`. This is function composition — the output tensor of one layer becomes the input tensor of the next.

**Analogy:** In a multi-stage RF amplifier chain, the output signal of stage N feeds directly into stage N+1. Each stage applies a gain and a non-linearity (saturation). That's exactly what these layers do.

### Step 4 — Define the Output Layer

Binary classification → one output unit with sigmoid activation. Output is a probability ∈ (0, 1).

$$\hat{y} = \sigma(W_3 \cdot h_2 + b_3) = \frac{1}{1 + e^{-(W_3 \cdot h_2 + b_3)}}$$

In [4]:
output_layer = Dense(1, activation='sigmoid')(hidden_layer2)

### Step 5 — Create the Model

Now I bind the input and output together. Keras traces the graph automatically — it knows every layer in between.

In [5]:
model = Model(inputs=input_layer, outputs=output_layer)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,569 (21.75 KB)

 Trainable params: 5,569 (21.75 KB)

 Non-trainable params: 0 (0.00 B)

`model.summary()` prints the architecture: layer names, output shapes, and parameter counts. Let me verify the parameter count manually:

| Layer | Weights | Biases | Total Params |
|-------|---------|--------|--------------|
| Dense(64) from input(20) | $20 \times 64 = 1280$ | $64$ | **1,344** |
| Dense(64) from Dense(64) | $64 \times 64 = 4096$ | $64$ | **4,160** |
| Dense(1) from Dense(64) | $64 \times 1 = 64$ | $1$ | **65** |
| **Total** | | | **5,569** |

### Step 6 — Compile

Before training I need to specify:
- **Optimizer:** Adam — adaptive learning rate, works well out of the box
- **Loss:** binary cross-entropy — the standard for binary classification
- **Metric:** accuracy — human-readable performance indicator

In [6]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### Step 7 — Train

I generate synthetic data here — 1,000 samples, 20 features each, binary labels. In a real project this would be real-world signal data, network metrics, or sensor readings.

In [7]:
# Synthetic training data
X_train = np.random.rand(1000, 20)
y_train = np.random.randint(2, size=(1000, 1))

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

Epoch 1/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4744 - loss: 0.6964 - val_accuracy: 0.4400 - val_loss: 0.7025
Epoch 2/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5422 - loss: 0.6876 - val_accuracy: 0.5400 - val_loss: 0.6933
Epoch 3/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5589 - loss: 0.6843 - val_accuracy: 0.4600 - val_loss: 0.7056
Epoch 4/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5722 - loss: 0.6795 - val_accuracy: 0.4600 - val_loss: 0.7071
Epoch 5/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5689 - loss: 0.6776 - val_accuracy: 0.4600 - val_loss: 0.7053
Epoch 6/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6100 - loss: 0.6733 - val_accuracy: 0.4800 - val_loss: 0.7015
Epoch 7/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5811 - loss: 0.6716 - val_accuracy: 0.4500 - val_loss: 0.7164
Epoch 8/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6167 - loss: 0.6663 - val_accuracy: 0.4600 - val_loss:

`model.fit` runs **gradient descent** over the data:
- Each **epoch** = one full pass over the training set
- Each **batch** = a mini-slice of 32 samples; gradients are computed and weights updated per batch

With random labels the model can't learn anything real — accuracy hovers near 50%. This is expected; with real structured data it would converge.

### Step 8 — Evaluate

In [8]:
# Synthetic test data
X_test = np.random.rand(200, 20)
y_test = np.random.randint(2, size=(200, 1))

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'🎯 Test loss:     {loss:.4f}')
print(f'✅ Test accuracy: {accuracy:.4f}')

🎯 Test loss:     0.7122
✅ Test accuracy: 0.4800


## Part 2 — ⚙️ Regularisation: Dropout & Batch Normalisation

Two techniques that dramatically improve training quality — especially on real-world, noisy data.

### 🔄 Dropout

During training, Dropout randomly **zeros out** a fraction $p$ of neuron activations at each step:

$$\tilde{h} = h \odot \text{Bernoulli}(1-p)$$

This forces the network to not rely on any single neuron — it has to distribute information across many paths. The result: better generalisation.

**Telecom analogy:** Like frequency diversity in radio links — if one sub-carrier fades, the others carry the message. No single path is essential, so the system is robust.

> ⚠️ Dropout is **only active during training**. During inference it's automatically disabled — all neurons contribute.

### 📊 Batch Normalisation

Batch Norm normalises the activations of each layer across the mini-batch, then re-scales with learned parameters $\gamma$ and $\beta$:

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \quad y_i = \gamma \hat{x}_i + \beta$$

**Effect:** Reduces *internal covariate shift* — the distribution of activations stays stable across layers, so I can use higher learning rates and training converges faster.

**Telecom analogy:** Like AGC (Automatic Gain Control) in a receiver chain — it normalises the signal level before each processing stage so the downstream filter always sees a consistent amplitude range.

### 🔄 Model with Dropout

In [9]:
from tensorflow.keras.layers import Dropout

# Input
input_layer = Input(shape=(20,))

# Hidden layer → Dropout
hidden_layer = Dense(64, activation='relu')(input_layer)
dropout_layer = Dropout(rate=0.5)(hidden_layer)

# Second hidden layer → Dropout
hidden_layer2 = Dense(64, activation='relu')(dropout_layer)
dropout_layer2 = Dropout(rate=0.5)(hidden_layer2)

# Output
output_layer = Dense(1, activation='sigmoid')(dropout_layer2)

model_dropout = Model(inputs=input_layer, outputs=output_layer)
model_dropout.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,569 (21.75 KB)

 Trainable params: 5,569 (21.75 KB)

 Non-trainable params: 0 (0.00 B)

### 📊 Model with Batch Normalisation

In [10]:
from tensorflow.keras.layers import BatchNormalization

# Input
input_layer = Input(shape=(20,))

# Hidden layer → BatchNorm
hidden_layer = Dense(64, activation='relu')(input_layer)
batch_norm_layer = BatchNormalization()(hidden_layer)

# Second hidden layer → BatchNorm
hidden_layer2 = Dense(64, activation='relu')(batch_norm_layer)
batch_norm_layer2 = BatchNormalization()(hidden_layer2)

# Output
output_layer = Dense(1, activation='sigmoid')(batch_norm_layer2)

model_batchnorm = Model(inputs=input_layer, outputs=output_layer)
model_batchnorm.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,081 (23.75 KB)

 Trainable params: 5,825 (22.75 KB)

 Non-trainable params: 256 (1.00 KB)

**Why Batch Norm adds parameters:** For each of the 64 units in each Dense layer, Batch Norm learns $\gamma$ and $\beta$ (scale and shift) — plus maintains running mean and variance for inference. That's $4 \times 64 = 256$ additional parameters per BN layer ($\gamma$, $\beta$, moving mean, moving variance).

## Part 3 — 🧪 Practice Exercises

> Data note: I use synthetic random data here. On random labels, accuracy ≈ 50% is correct behaviour — there's no real pattern to learn. What matters is that the training runs without errors and the architecture is right.

### Exercise 1 — 🔄 Add Dropout Layers

**Goal:** Build a model with Dropout (rate = 0.5) after each hidden layer. Compile, train, and evaluate.

In [11]:
from tensorflow.keras.layers import Dropout, Input, Dense
from tensorflow.keras.models import Model

# Input layer
input_layer = Input(shape=(20,))

# Hidden layer 1 + Dropout
hidden_layer1 = Dense(64, activation='relu')(input_layer)
dropout1 = Dropout(0.5)(hidden_layer1)

# Hidden layer 2 + Dropout
hidden_layer2 = Dense(64, activation='relu')(dropout1)
dropout2 = Dropout(0.5)(hidden_layer2)

# Output layer
output_layer = Dense(1, activation='sigmoid')(dropout2)

# Build model
model_ex1 = Model(inputs=input_layer, outputs=output_layer)
model_ex1.summary()

# Compile
model_ex1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train
model_ex1.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

# Evaluate
loss, accuracy = model_ex1.evaluate(X_test, y_test, verbose=0)
print(f'🎯 Test loss:     {loss:.4f}')
print(f'✅ Test accuracy: {accuracy:.4f}')

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,569 (21.75 KB)

 Trainable params: 5,569 (21.75 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 694us/step - accuracy: 0.4790 - loss: 0.7288
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 746us/step - accuracy: 0.4910 - loss: 0.7130
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 819us/step - accuracy: 0.5270 - loss: 0.6967
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 590us/step - accuracy: 0.5430 - loss: 0.6978
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 608us/step - accuracy: 0.4880 - loss: 0.7053
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 601us/step - accuracy: 0.5040 - loss: 0.6967
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 584us/step - accuracy: 0.5050 - loss: 0.6986
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 595us/step - accuracy: 0.4980 - loss: 0.6993
Epoch 9/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 612us/step - accuracy: 0.5230 - loss: 0.6943
Epoch 10/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 605us/step - accuracy: 0.5230 - loss: 0.6927
🎯 Test loss:     0.6947
✅ Test accuracy: 0.4550


**What I observe:** With Dropout at rate 0.5, exactly half the neurons are masked at each training step. The model trains more slowly but generalises better. On random data the accuracy is still ~50% — that's correct.

### Exercise 2 — ➕ Change Activation Functions to Tanh

**Goal:** Replace ReLU with Tanh in hidden layers. Tanh outputs values in $(-1, 1)$ — zero-centred, which can help gradient flow in shallow networks.

$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$$

In [12]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

# Input layer
input_layer = Input(shape=(20,))

# Hidden layers with Tanh
hidden_layer1 = Dense(64, activation='tanh')(input_layer)
hidden_layer2 = Dense(64, activation='tanh')(hidden_layer1)

# Output layer
output_layer = Dense(1, activation='sigmoid')(hidden_layer2)

# Build model
model_ex2 = Model(inputs=input_layer, outputs=output_layer)
model_ex2.summary()

# Compile
model_ex2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train
model_ex2.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

# Evaluate
loss, accuracy = model_ex2.evaluate(X_test, y_test, verbose=0)
print(f'🎯 Test loss:     {loss:.4f}')
print(f'✅ Test accuracy: {accuracy:.4f}')

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,569 (21.75 KB)

 Trainable params: 5,569 (21.75 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 661us/step - accuracy: 0.5230 - loss: 0.7020
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 628us/step - accuracy: 0.4980 - loss: 0.7016
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 630us/step - accuracy: 0.5310 - loss: 0.6876
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 604us/step - accuracy: 0.5360 - loss: 0.6902
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 588us/step - accuracy: 0.5390 - loss: 0.6876
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 578us/step - accuracy: 0.5630 - loss: 0.6871
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 606us/step - accuracy: 0.5540 - loss: 0.6892
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 581us/step - accuracy: 0.5630 - loss: 0.6864
Epoch 9/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 586us/step - accuracy: 0.5620 - loss: 0.6836
Epoch 10/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 577us/step - accuracy: 0.5420 - loss: 0.6897
🎯 Test loss:     0.7299
✅ Test accuracy: 0.4900


**ReLU vs Tanh comparison:**

| Property | ReLU | Tanh |
|----------|------|------|
| Output range | $[0, \infty)$ | $(-1, 1)$ |
| Zero-centred | ❌ | ✅ |
| Vanishing gradient | At negative inputs | At extremes ($\pm$large $z$) |
| Dying neurons | Possible (dead ReLU) | No |
| Computationally | Very fast | Slightly slower |
| Default choice | Deep networks | Shallower networks, RNNs |

### Exercise 3 — 📊 Add Batch Normalisation

**Goal:** Add BatchNormalization after each hidden layer to stabilise training.

In [ ]:
from tensorflow.keras.layers import BatchNormalization, Input, Dense
from tensorflow.keras.models import Model

# Input layer
input_layer = Input(shape=(20,))

# Hidden layer 1 + BatchNorm
hidden_layer1 = Dense(64, activation='relu')(input_layer)
batch_norm1 = BatchNormalization()(hidden_layer1)

# Hidden layer 2 + BatchNorm
hidden_layer2 = Dense(64, activation='relu')(batch_norm1)
batch_norm2 = BatchNormalization()(hidden_layer2)

# Output layer
output_layer = Dense(1, activation='sigmoid')(batch_norm2)

# Build model
model_ex3 = Model(inputs=input_layer, outputs=output_layer)
model_ex3.summary()

# Compile
model_ex3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train
model_ex3.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

# Evaluate
loss, accuracy = model_ex3.evaluate(X_test, y_test, verbose=0)
print(f'🎯 Test loss:     {loss:.4f}')
print(f'✅ Test accuracy: {accuracy:.4f}')

## 📊 Summary

### What I built in this notebook

| Step | What I did | Key concept |
|------|------------|-------------|
| 1 | Imported TF/Keras | `Model`, `Input`, `Dense` |
| 2 | Defined input | `Input(shape=(20,))` |
| 3 | Wired hidden layers | Functional chaining: `layer(prev_tensor)` |
| 4 | Output layer | `Dense(1, activation='sigmoid')` |
| 5 | Built model | `Model(inputs=..., outputs=...)` |
| 6 | Compiled | Adam optimizer, binary cross-entropy |
| 7 | Trained | `model.fit(X, y, epochs, batch_size)` |
| 8 | Evaluated | `model.evaluate(X_test, y_test)` |
| 9 | Dropout | Regularisation via random neuron masking |
| 10 | Batch Norm | Normalise activations per mini-batch |

### Regularisation techniques compared

| Technique | When to use | Effect | Adds params? |
|-----------|------------|--------|--------------|
| **Dropout** | Overfitting | Forces distributed representations | ❌ No |
| **Batch Norm** | Deep networks, unstable training | Stabilises activations, faster convergence | ✅ Yes ($\gamma, \beta$) |
| **Both** | Large datasets, deep models | Best of both worlds | ✅ Yes |

### Functional API vs Sequential

| | Sequential | Functional |
|-|------------|------------|
| Multi-input models | ❌ | ✅ |
| Multi-output models | ❌ | ✅ |
| Skip connections | ❌ | ✅ |
| Shared layers | ❌ | ✅ |
| Readability (simple case) | ✅ | ✅ |

## 🧪 Sandbox

Ideas I want to explore from here:

1. **Multi-input model** — e.g., two separate input streams (time-domain + frequency-domain features) merged at a Dense layer
2. **Skip connections** — add the input directly to a later layer's input (like ResNets)
3. **Real data** — replace the synthetic arrays with the MNIST or a telecom KPI dataset
4. **Visualise training curves** — plot loss and accuracy vs epoch with matplotlib

In [ ]:
# 🧪 Sandbox: multi-input model example
from tensorflow.keras.layers import concatenate

# Two separate input streams
input_a = Input(shape=(20,), name='time_domain')
input_b = Input(shape=(20,), name='freq_domain')

# Each stream gets its own hidden layer
branch_a = Dense(32, activation='relu')(input_a)
branch_b = Dense(32, activation='relu')(input_b)

# Merge the two branches
merged = concatenate([branch_a, branch_b])

# Shared classification head
hidden = Dense(64, activation='relu')(merged)
output = Dense(1, activation='sigmoid')(hidden)

model_multi = Model(inputs=[input_a, input_b], outputs=output)
model_multi.summary()

In [ ]:
# 🧪 Sandbox: visualise training history
import matplotlib.pyplot as plt

# Retrain baseline model and capture history
input_layer = Input(shape=(20,))
h1 = Dense(64, activation='relu')(input_layer)
h2 = Dense(64, activation='relu')(h1)
out = Dense(1, activation='sigmoid')(h2)
model_vis = Model(inputs=input_layer, outputs=out)
model_vis.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

hist = model_vis.fit(X_train, y_train, epochs=20, batch_size=32,
                     validation_split=0.1, verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist.history['loss'], label='Train loss')
axes[0].plot(hist.history['val_loss'], label='Val loss')
axes[0].set_title('📈 Loss over epochs')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(hist.history['accuracy'], label='Train acc')
axes[1].plot(hist.history['val_accuracy'], label='Val acc')
axes[1].set_title('✅ Accuracy over epochs')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()